In [1]:
using Pkg
Pkg.activate(@__DIR__)

  Activating project at `c:\Users\jerem\OneDrive\Desktop\GitHub\TrustRegionRadius.jl\benchmark`


In [6]:
# Auto-develop the parent package if it is not yet in the environment
if !haskey(Pkg.project().dependencies, "TrustRegionRadius")
    @info "Developing TrustRegionRadius from parent directory…"
    Pkg.develop(PackageSpec(path = joinpath(@__DIR__, "..")))
end

┌ Info: Developing TrustRegionRadius from parent directory…
└ @ Main c:\Users\jerem\OneDrive\Desktop\GitHub\TrustRegionRadius.jl\benchmark\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W1sZmlsZQ==.jl:3
   Resolving package versions...
    Updating `C:\Users\jerem\OneDrive\Desktop\GitHub\TrustRegionRadius.jl\benchmark\Project.toml`
⌃ [1b53aba6] ↓ CUTEst v1.3.8 ⇒ v1.3.7
  [b2e96d2e] + TrustRegionRadius v0.1.0 `c:\Users\jerem\OneDrive\Desktop\GitHub\TrustRegionRadius.jl\benchmark\..`
    Updating `C:\Users\jerem\OneDrive\Desktop\GitHub\TrustRegionRadius.jl\benchmark\Manifest.toml`
  [a4c015fc] + ANSIColoredPrinters v0.0.1
  [1520ce14] + AbstractTrees v0.4.5
  [66dad0bd] + AliasTables v1.1.3
  [dce04be8] + ArgCheck v2.5.0
  [ecbce9bc] + BenchmarkProfiles v0.4.6
  [d1d4a3ce] + BitFlags v0.1.10
  [336ed68f] + CSV v0.10.16
⌃ [1b53aba6] ↓ CUTEst v1.3.8 ⇒ v1.3.7
  [944b1d66] + CodecZlib v0.7.8
  [35d6a980] + ColorSchemes v3.31.0
  [3da002f7] + ColorTypes v0.12.1
  [c3611d14] + ColorVectorSp

In [8]:
Pkg.add("Printf")
Pkg.add("Plots")
Pkg.add("JLD2")
Pkg.add("Statistics")

   Resolving package versions...
    Updating `C:\Users\jerem\OneDrive\Desktop\GitHub\TrustRegionRadius.jl\benchmark\Project.toml`
  [de0858da] + Printf v1.11.0
  No Changes to `C:\Users\jerem\OneDrive\Desktop\GitHub\TrustRegionRadius.jl\benchmark\Manifest.toml`
   Resolving package versions...
    Updating `C:\Users\jerem\OneDrive\Desktop\GitHub\TrustRegionRadius.jl\benchmark\Project.toml`
  [91a5bcdd] + Plots v1.41.6
  No Changes to `C:\Users\jerem\OneDrive\Desktop\GitHub\TrustRegionRadius.jl\benchmark\Manifest.toml`
   Resolving package versions...
    Updating `C:\Users\jerem\OneDrive\Desktop\GitHub\TrustRegionRadius.jl\benchmark\Project.toml`
⌅ [033835bb] + JLD2 v0.5.15
  No Changes to `C:\Users\jerem\OneDrive\Desktop\GitHub\TrustRegionRadius.jl\benchmark\Manifest.toml`
   Resolving package versions...
    Updating `C:\Users\jerem\OneDrive\Desktop\GitHub\TrustRegionRadius.jl\benchmark\Project.toml`
  [10745b16] + Statistics v1.11.1
  No Changes to `C:\Users\jerem\OneDrive\Desktop\

In [9]:
Pkg.status()

Status `C:\Users\jerem\OneDrive\Desktop\GitHub\TrustRegionRadius.jl\benchmark\Project.toml`
  [6e4b80f9] BenchmarkTools v1.8.0
⌃ [1b53aba6] CUTEst v1.3.7
⌅ [033835bb] JLD2 v0.5.15
  [91a5bcdd] Plots v1.41.6
  [10745b16] Statistics v1.11.1
  [b2e96d2e] TrustRegionRadius v0.1.0 `c:\Users\jerem\OneDrive\Desktop\GitHub\TrustRegionRadius.jl\benchmark\..`
  [37e2e46d] LinearAlgebra v1.11.0
  [de0858da] Printf v1.11.0
Info Packages marked with ⌃ and ⌅ have new versions available. Those with ⌃ may be upgradable, but those with ⌅ are restricted by compatibility constraints from upgrading. To see why use `status --outdated`


In [10]:
Pkg.instantiate()

In [11]:
using Revise
using TrustRegionRadius
using CUTEst
using JLD2
using LinearAlgebra
using Printf

In [12]:
const SOLVER_PARAMS = TRSolverParams(
    η₁ = 0.1,
    η₂ = 0.9,
    Δ₀ = 1.0,
    max_iterations = 10_000,
    tol = 1e-5,
)

TRSolverParams{Float64}:
  η₁: 0.1  η₂: 0.9
  Δ₀: 1.0
  max_iterations: 10000
  tol: 1.0e-5


In [13]:
# Factory functions so each run gets a fresh (and for R4 mutable) rule
const RULES = [
    ("R1", () -> R1ClassicalUpdate(0.25, 0.50, 2.0)),
    ("R2", () -> R2StepSizeUpdate(0.25, 0.80, 2.0)),
    ("R3", () -> R3DFOLikeUpdate(0.25, 0.50, 2.0, 1.0)),
    ("R4", () -> R4RelativeGradUpdate(0.25, 2.0,  1.0)),
]

4-element Vector{Tuple{String, Function}}:
 ("R1", var"#15#19"())
 ("R2", var"#16#20"())
 ("R3", var"#17#21"())
 ("R4", var"#18#22"())

In [14]:
try
    finalize(nlp) 
catch e
    @error "No model to finalize: $e"
end

┌ Error: No model to finalize: UndefVarError(:nlp, Main)
└ @ Main c:\Users\jerem\OneDrive\Desktop\GitHub\TrustRegionRadius.jl\benchmark\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sZmlsZQ==.jl:4


In [15]:
# =============================================================================
# Problem selection
# =============================================================================

@info "Querying CUTEst problem list…"

prob_name = "ROSENBR"
nlp = CUTEstModel(prob_name)

┌ Info: Querying CUTEst problem list…
└ @ Main c:\Users\jerem\OneDrive\Desktop\GitHub\TrustRegionRadius.jl\benchmark\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:5


  Problem name: ROSENBR
   All variables: ████████████████████ 2      All constraints: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
            free: ████████████████████ 2                 free: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
           lower: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0                lower: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
           upper: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0                upper: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
         low/upp: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0              low/upp: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
           fixed: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0                fixed: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
          infeas: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0               infeas: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
            nnzh: (  0.00% sparsity)   3               linear: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
                                                    nonlinear: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
                                                         nnzj: (------% sparsity)         
                                                     lin_nnzj: (--

In [16]:
# Safety check: skip if problem has constraints or wrong size
@show nvar = nlp.meta.nvar
@show ncon = nlp.meta.ncon
if nvar < 2 || nvar > 500 || ncon > 0
    @warn "  $rule_name: skipping $prob_name (nvar=$nvar, ncon=$ncon)"
    finalize(nlp)
    nlp = nothing
end

nvar = nlp.meta.nvar = 2
ncon = nlp.meta.ncon = 0


In [17]:
rule_number = 3
rule = RULES[rule_number][2]()

R3 DFO-Like Update:
  γ₁ (contraction):   0.25
  γ₂ (no-expand):     0.5
  γ₃ (expansion):     2.0
  ζ  (threshold):     1.0


In [18]:
SOLVER_PARAMS

TRSolverParams{Float64}:
  η₁: 0.1  η₂: 0.9
  Δ₀: 1.0
  max_iterations: 10000
  tol: 1.0e-5


In [21]:
subsolver = SteihaugTointCG()

SteihaugTointCG(0.1, 0.5, 100)

In [43]:
stats = trust_region_radius(nlp ; 
    rule = rule,
    subsolver = subsolver,
    params = SOLVER_PARAMS,
)

"Execution stats: first-order stationary"

In [40]:
@show stats.iter
@show stats.objective
@show stats.status
@show stats.elapsed_time ;

stats.iter = 37
stats.objective = 2.1671363674499844e-12
stats.status = :first_order
stats.elapsed_time = 0.0010001659393310547


In [45]:
print(stats)

Generic Execution stats
  status: first-order stationary
  objective value: 2.1671363674499844e-12
  primal feasibility: 0.0
  dual feasibility: 1.315656424949353e-6
  solution: [0.9999985290548208  0.9999970522324795]
  iterations: 37
  elapsed time: 0.0


In [19]:
t0  = time()
out = trust_region_solver(nlp, rule, SOLVER_PARAMS)
elapsed = time() - t0

UndefVarError: UndefVarError: `trust_region_solver` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
out

In [ ]:
out.delta_trajectory